# Session 5 (Day 3 — Morning)
## Visualisation, Dashboards & AI-Assisted Development

GIS Executive Seminar — Enhancing Data Analysis Skills for Informed Investment Insights

---
> Guiding question: "How do I communicate what the data is telling me — and build a tool that does it automatically?"

### Learning Objectives
- Understand the five principles of effective financial chart design
- Build interactive charts using Plotly (hover labels, range selectors, legend toggle)
- Assemble a multi-panel dashboard from individual Plotly charts
- Export an interactive dashboard as a self-contained HTML file
- Understand what Claude Code is and how to brief it effectively
- Practise writing Claude Code instructions in the style of a senior investment professional

### How to Use This Notebook
Run all cells in order. The setup section re-downloads the GIS data and recomputes all metrics — this takes about 30 seconds. The Plotly charts are interactive: hover over lines, click legend items to show/hide, and use the range selector buttons.

## Note on Self-Contained Sessions

This notebook re-downloads all data and recomputes all metrics from Sessions 1–4 in a condensed setup section. This takes about 30 seconds. Every session is self-contained — you do not need to have run any previous notebook.

In [ ]:
# Install all required libraries
# yfinance   — download price and financial data from Yahoo Finance
# fredapi    — download macroeconomic data from the Federal Reserve (FRED)
# plotly     — interactive charts (the main new library in this session)
# scipy      — scientific computing: we use it for the normal distribution in VaR
# seaborn    — used only for one 'before' comparison cell showing matplotlib-style heatmap
# numpy      — numerical computing (arrays, maths operations)
# pandas     — data manipulation (DataFrames, time series)
!pip install yfinance fredapi plotly scipy seaborn --quiet
print('Libraries installed.')

In [ ]:
import pandas as pd                          # data manipulation — DataFrames and time series
import numpy as np                           # numerical computing — arrays and maths
import matplotlib.pyplot as plt             # still used for the 'before' example in the chart principles section
import seaborn as sns                        # used for the matplotlib correlation heatmap comparison
import plotly.graph_objects as go           # plotly's core chart object — like plt.figure() but interactive
from plotly.subplots import make_subplots   # create multi-panel dashboards
import plotly.figure_factory as ff          # plotly's factory for annotated heatmaps
from scipy import stats                      # normal distribution for parametric VaR
from fredapi import Fred                     # FRED API client
import yfinance as yf                        # Yahoo Finance data downloader
from datetime import date                    # date arithmetic
from IPython.display import display          # display DataFrames nicely in Colab

print('All libraries imported.')
print(f'Plotly version: {go.__version__}')

In [ ]:
# ── FRED API KEY ──────────────────────────────────────────────────────────────
# Replace 'YOUR_FRED_API_KEY' with your real key from https://fred.stlouisfed.org/docs/api/api_key.html
# Registration is free and takes about 2 minutes.
FRED_API_KEY = 'YOUR_FRED_API_KEY'   # ← CHANGE ME

In [ ]:
# ── STEP 1: DATE RANGE ───────────────────────────────────────────────────────
PORTFOLIO_TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'GLD', 'IEF']   # the 7 GIS holdings
BENCHMARK_TICKER  = '^GSPC'                                                   # S&P 500 — benchmark only
ALL_TICKERS       = PORTFOLIO_TICKERS + [BENCHMARK_TICKER]                   # combined list for download
n_assets          = len(PORTFOLIO_TICKERS)                                   # number of holdings = 7
weights           = np.repeat(1 / n_assets, n_assets)                       # equal weight: each holding = 1/7

end_date   = date.today()                                                     # today's date
start_date = date(end_date.year - 3, end_date.month, end_date.day)          # exactly 3 years ago

# ── STEP 2: DOWNLOAD PRICES ───────────────────────────────────────────────────
print('Downloading GIS prices from Yahoo Finance...')
raw_data   = yf.download(                  # download price data from Yahoo Finance
    tickers      = ALL_TICKERS,            # download all 8 tickers at once
    start        = start_date,             # start of the 3-year window
    end          = end_date,               # today
    auto_adjust  = True,                   # adjust prices for splits and dividends automatically
    progress     = True,                   # show a download progress bar
)
all_prices = raw_data['Close'].ffill()     # extract closing prices; forward-fill any missing values
prices     = all_prices[PORTFOLIO_TICKERS]  # price DataFrame for the 7 portfolio holdings
benchmark  = all_prices[[BENCHMARK_TICKER]]  # price Series for the S&P 500 benchmark

# ── STEP 3: RETURNS ───────────────────────────────────────────────────────────
daily_returns          = prices.pct_change().dropna()                        # daily % change for each holding
portfolio_daily_return = daily_returns.dot(weights)                          # weighted average = portfolio return
benchmark_daily_return = benchmark.pct_change().dropna().squeeze()           # S&P 500 daily return as a Series

# ── STEP 4: CUMULATIVE RETURN INDEX ──────────────────────────────────────────
cumulative_portfolio  = (1 + portfolio_daily_return).cumprod().mul(100)     # portfolio grows from 100
cumulative_benchmark  = (1 + benchmark_daily_return).cumprod().mul(100)     # benchmark grows from 100
cumulative_individual = (1 + daily_returns).cumprod().mul(100)              # each holding grows from 100

# ── STEP 5: ROLLING 20-DAY VOLATILITY ────────────────────────────────────────
rolling_vol = (
    daily_returns
    .rolling(window=20)     # look at the last 20 trading days
    .std()                  # standard deviation of those 20 daily returns
    .mul(np.sqrt(252))      # annualise: multiply by sqrt(252 trading days per year)
    .dropna()               # remove the first 19 rows where we don't have 20 days yet
)

# ── STEP 6: MONTHLY RETURNS MATRIX (for correlation) ─────────────────────────
monthly_returns = (
    prices
    .resample('ME')         # group trading days into calendar months
    .last()                 # take the last trading day's price as the month-end price
    .pct_change()           # month-over-month return
    .dropna()               # remove the first row (no previous month to compare against)
)
corr_matrix = monthly_returns.corr().round(2)   # 7×7 correlation matrix, rounded to 2 decimal places

# ── STEP 7: PORTFOLIO PERFORMANCE METRICS ────────────────────────────────────
ann_return    = portfolio_daily_return.mean() * 252    # annualised return: average daily return × 252 trading days
ann_vol       = portfolio_daily_return.std() * np.sqrt(252)   # annualised volatility

running_max   = cumulative_portfolio.cummax()          # highest value reached up to each date
drawdown      = cumulative_portfolio.div(running_max).sub(1)   # % below the prior peak
max_dd        = drawdown.min()                         # the worst drawdown in the period

print(f'\nSetup complete.')
print(f'  Prices:         {prices.shape}')
print(f'  Ann. Return:    {ann_return:.2%}')
print(f'  Ann. Volatility:{ann_vol:.2%}')
print(f'  Max Drawdown:   {max_dd:.2%}')

In [ ]:
# ── STEP 8: MACRO DATA FROM FRED ─────────────────────────────────────────────
fred = Fred(api_key=FRED_API_KEY)   # create a FRED client using your API key

# Download the 2-year / 10-year Treasury yield spread
# Positive = normal (long rates > short rates); Negative = inversion (often precedes recession)
spread_series = (
    fred.get_series(
        'T10Y2Y',                            # FRED series code for the 10Y minus 2Y spread
        observation_start = start_date,      # start of our 3-year window
        observation_end   = end_date,        # today
    )
    .dropna()                                # remove any missing values
)

# Download NBER recession indicator: 1 = recession month, 0 = expansion month
usrec = fred.get_series(
    'USREC',
    observation_start = start_date,
    observation_end   = end_date,
)

# Find recession start dates: months where the value switches from 0 to 1
rec_starts = usrec[(usrec == 1) & (usrec.shift(1) == 0)].index
# Find recession end dates: months where the value switches from 1 to 0
rec_ends   = usrec[(usrec == 0) & (usrec.shift(1) == 1)].index

# ── MONTE CARLO SIMULATION (recompute for this session) ──────────────────────
# 500 paths, 252 trading days (1 year forward)
# Each path: draw a random daily return vector from a multivariate normal distribution
#            that matches the historical mean and covariance of the 7 holdings
np.random.seed(42)       # fix random seed for reproducibility
n_sim  = 500             # number of simulation paths
n_days = 252             # number of trading days to simulate (1 year)
mean_r = daily_returns.mean().values    # historical mean daily return for each of the 7 holdings
cov_r  = daily_returns.cov().values     # historical covariance matrix (captures correlations between holdings)

# Draw random daily returns: shape is (n_days, n_sim)
# Each column is one simulated path; each row is one trading day
sim_returns = np.random.multivariate_normal(mean_r, cov_r, size=(n_days, n_sim))

# Compute the equally-weighted portfolio return for each path on each day
sim_port = np.dot(sim_returns, weights)   # shape: (n_days, n_sim)

# Convert daily returns to a cumulative index starting at 100
sim_cum = (1 + sim_port).cumprod(axis=0).T * 100   # shape: (n_sim, n_days)

# Extract percentile bands across all simulation paths
p5  = np.percentile(sim_cum, 5,  axis=0)   # worst 5% of outcomes at each day
p25 = np.percentile(sim_cum, 25, axis=0)   # 25th percentile
p50 = np.percentile(sim_cum, 50, axis=0)   # median path
p75 = np.percentile(sim_cum, 75, axis=0)   # 75th percentile
p95 = np.percentile(sim_cum, 95, axis=0)   # best 5% of outcomes at each day

print('FRED data and Monte Carlo ready.')

---
## Section 1: Financial Visualisation Principles

A chart is not just a picture — it is a communication tool. Investment committees, portfolio managers, and clients all form opinions based on how data is presented. A poorly designed chart can obscure the very insight it is supposed to convey. These five principles are drawn from the practice of institutional investment communication.

---

### Principle 1 — Label Everything

Axis labels, chart titles, units, and data sources must be present on every chart. A chart without a unit on the y-axis is not a chart — it is a picture. Is the y-axis showing *percentage*, *index value*, *dollar amount*, or *basis points*? A viewer cannot know without a label.

Investment committees have sent analysts back to redo work because of missing units. The standard in professional investment communication is: if you cannot immediately answer "what is the unit of this axis?", the label is missing.

**What to look for in your output:** Does every axis have a title with a unit? Does the chart title include the metric being shown?

---

### Principle 2 — Match the Chart Type to the Data Type

- **Time series** (how something changes over time) → **line chart**
- **Composition** (portfolio weights, sector splits, asset allocation) → **bar chart** or **pie chart** (max 5 segments)
- **Distribution** (how returns are spread across values) → **histogram**
- **Relationships** (does asset A move with asset B?) → **scatter plot**
- **Ranking** (which holding had the highest return this month?) → **horizontal bar chart**

Never use a pie chart for more than 5 segments — human eyes cannot distinguish 7+ slices accurately. A bar chart always communicates composition more precisely.

**What to look for in your output:** Are you choosing the right chart for the question? A correlation matrix should be a heatmap, not a line chart.

---

### Principle 3 — Colour for Signal, Not Decoration

Use colour to convey information:
- **Red** = loss, risk, warning, below zero
- **Green** = gain, positive, above threshold
- **Grey** = neutral, reference, background
- **Blue** = primary series, portfolio, the thing you are analysing

Do not use 8 different shades of blue "to make it look nice". Do not use bright orange and hot pink because they are available. Accessibility note: approximately 8% of men have red-green colour blindness. Where possible, use blue-orange instead of red-green for pairwise comparisons.

**What to look for in your output:** Can a viewer immediately identify what is positive vs negative, important vs background, without reading the legend?

---

### Principle 4 — Beware the Dual Axis

Two y-axes with different scales can make any two series look correlated — or uncorrelated — depending entirely on how you set the scale. This is one of the most common ways to mislead with charts, whether intentionally or accidentally.

If you must use a dual axis (e.g., comparing a return index to a yield), label both axes clearly, state the scale difference, and consider whether a second chart would communicate more honestly.

Many professional investment style guides (including those of major asset managers) prohibit dual-axis charts entirely in client-facing materials.

**What to look for in your output:** Are you showing two series on the same axis that have incompatible units? If so, a dual axis or a separate panel is more honest than normalising them to the same scale.

---

### Principle 5 — Write Conclusion-First Titles

| Weak (describes the chart) | Strong (tells the reader what to see) |
|---|---|
| Cumulative Return Chart | GIS Portfolio Has Outperformed the S&P 500 Since 2022 |
| Rolling Volatility | Technology Holdings Drive Volatility Spikes in the Portfolio |
| Correlation Matrix | GLD and IEF Provide Genuine Diversification Against Equities |
| Yield Spread Chart | Yield Curve Inversion Has Historically Preceded Recessions by 12–18 Months |

The title should tell the reader what to conclude — not just describe what is being plotted. Investment committee presentations are more effective when charts lead with the insight rather than leaving the reader to find it.

---

### Static vs Interactive Charts

| Format | Tool | When to Use |
|--------|------|-------------|
| **Static PNG/PDF** | matplotlib | Formal reports, presentations, PDFs, emails to large audiences where recipients cannot interact with files |
| **Interactive HTML** | Plotly | Analysis sessions, internal dashboards, client meetings where exploration is expected |

Plotly produces **both** from the same code:
- `fig.show()` → interactive chart rendered in the browser or Colab
- `fig.write_image('chart.png')` → static PNG (requires the `kaleido` package)
- `fig.write_html('chart.html')` → self-contained interactive HTML file

This flexibility means you do not have to choose one tool for reports and another for analysis — Plotly handles both.

---
## From matplotlib to Plotly — Why Switch?

Sessions 1–4 used **matplotlib** to build all charts. This was intentional: matplotlib is the foundational Python charting library, and understanding it makes every other charting library easier to learn. But matplotlib produces *static images*. Once rendered, the chart cannot be explored.

**What Plotly adds:**

| Feature | matplotlib | Plotly |
|---------|-----------|--------|
| Output format | Static PNG / SVG | Interactive HTML |
| Hover labels | Not available | Shows exact values, dates, custom text on hover |
| Legend toggle | Not available | Click to show/hide individual series |
| Time range zoom | Not available | Built-in range selector buttons (1M / 3M / 1Y / All) |
| Export to HTML | Not available | `fig.write_html()` — opens in any browser, no Python needed |
| Colab integration | Static image | Interactive widget rendered inline |

**Syntax differences:** Plotly uses `go.Figure()` instead of `plt.figure()`. The concepts are the same — you create a figure, add traces (series), and configure a layout. The main difference is that configuration is done via Python dictionaries passed as arguments, rather than function calls.

```python
# matplotlib style
fig, ax = plt.subplots()
ax.plot(dates, values, label='GIS Portfolio')
ax.set_title('Cumulative Return')
plt.show()

# Plotly equivalent
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=values, name='GIS Portfolio'))
fig.update_layout(title='Cumulative Return')
fig.show()
```

**Both are valid tools.** Use matplotlib when you need static images for reports or PDFs. Use Plotly when you need interactive exploration or shareable HTML deliverables. Most professional investment teams use both.

**Plotly in Colab:** Charts rendered via `fig.show()` appear inline as interactive widgets. You can zoom, pan, hover, and toggle legend items directly in the notebook output. They can also be saved to `.html` files and opened in any browser — no Python installation required on the recipient's machine.

In [ ]:
# ── CHART 1: CUMULATIVE RETURN (PLOTLY) ──────────────────────────────────────
# go.Figure() creates a blank Plotly figure — equivalent to fig, ax = plt.subplots()
fig1 = go.Figure()

# Add one trace (line) per portfolio holding
# A 'trace' in Plotly is equivalent to one series on a chart
for ticker in PORTFOLIO_TICKERS:
    fig1.add_trace(
        go.Scatter(
            x    = cumulative_individual.index,          # x-axis: date index
            y    = cumulative_individual[ticker],        # y-axis: cumulative return value for this ticker
            name = ticker,                               # legend label for this series
            mode = 'lines',                              # 'lines' = connected line (no dot at each data point)
            line = dict(width=1.5),                      # line thickness in pixels
            opacity = 0.7,                               # slight transparency so overlapping lines are visible
            hovertemplate = (                            # format of the tooltip shown when hovering
                f'<b>{ticker}</b><br>'                  # bold ticker name on the first line
                'Date: %{x|%Y-%m-%d}<br>'              # formatted date (YYYY-MM-DD)
                'Value: %{y:.1f}<extra></extra>'        # value rounded to 1 decimal; <extra></extra> removes the trace name from the tooltip
            ),
        )
    )

# Add the equally-weighted portfolio line — thicker so it stands out from individual holdings
fig1.add_trace(
    go.Scatter(
        x    = cumulative_portfolio.index,
        y    = cumulative_portfolio,
        name = 'GIS Portfolio (Equal Weight)',
        mode = 'lines',
        line = dict(width=3, color='black'),              # thick black line — the portfolio summary
        hovertemplate = '<b>GIS Portfolio</b><br>Date: %{x|%Y-%m-%d}<br>Value: %{y:.1f}<extra></extra>',
    )
)

# Add the S&P 500 benchmark line — dashed red to distinguish from portfolio holdings
fig1.add_trace(
    go.Scatter(
        x    = cumulative_benchmark.index,
        y    = cumulative_benchmark,
        name = 'S&P 500 Benchmark',
        mode = 'lines',
        line = dict(width=2.5, color='crimson', dash='dash'),   # dashed red = benchmark reference
        hovertemplate = '<b>S&P 500</b><br>Date: %{x|%Y-%m-%d}<br>Value: %{y:.1f}<extra></extra>',
    )
)

# Add a horizontal reference line at 100 — the starting value for all series
fig1.add_hline(
    y                   = 100,
    line_dash           = 'dot',           # dotted line style
    line_color          = 'grey',
    line_width          = 1,
    annotation_text     = 'Starting value (100)',
    annotation_position = 'bottom right',  # position the label at the right end of the line
)

# Update the overall layout — title, axis labels, range selectors, legend position
fig1.update_layout(
    title = dict(
        text = 'GIS Portfolio — Cumulative Return (Base = 100)',   # conclusion-first title
        font = dict(size=16),
    ),
    xaxis = dict(
        title = 'Date',
        rangeselector = dict(                  # adds clickable zoom buttons above the chart
            buttons = [
                dict(count=1,  label='1M',  step='month', stepmode='backward'),   # zoom to last 1 month
                dict(count=3,  label='3M',  step='month', stepmode='backward'),   # zoom to last 3 months
                dict(count=12, label='1Y',  step='month', stepmode='backward'),   # zoom to last 12 months
                dict(step='all', label='All'),                                     # show all data
            ]
        ),
        rangeslider = dict(visible=False),     # hide the scroll bar below the chart for a compact view
    ),
    yaxis     = dict(title='Growth of $100 Invested'),
    hovermode = 'x unified',                   # show all series values simultaneously when hovering on any date
    legend    = dict(orientation='v', x=1.01, y=1),   # legend to the right of the chart
    height    = 500,
    template  = 'plotly_white',                # clean white background — professional appearance
)

fig1.show()   # display the interactive chart inline in Colab

print()
print('Interactive controls:')
print('  Hover over any date to see all values simultaneously')
print('  Click a legend item to hide/show that series')
print('  Double-click a legend item to isolate that series')
print('  Use the 1M / 3M / 1Y / All buttons to zoom the time axis')

In [ ]:
# ── CHART 2: ROLLING 20-DAY VOLATILITY (PLOTLY) ─────────────────────────────
# Rolling volatility shows how much risk each holding is contributing at any point in time.
# Spikes indicate periods of market stress. If all holdings spike simultaneously,
# that suggests a systemic shock — diversification breaks down.

fig2 = go.Figure()   # create a new blank figure

# Add one trace per portfolio holding
for ticker in PORTFOLIO_TICKERS:
    fig2.add_trace(
        go.Scatter(
            x    = rolling_vol.index,
            y    = rolling_vol[ticker] * 100,    # multiply by 100 to show as percentage (e.g. 0.15 → 15%)
            name = ticker,
            mode = 'lines',
            line = dict(width=1.5),
            opacity = 0.8,                        # slight transparency for overlapping lines
            hovertemplate = (
                f'<b>{ticker}</b><br>'
                'Date: %{x|%Y-%m-%d}<br>'
                'Ann. Vol: %{y:.1f}%<extra></extra>'   # show annualised volatility as a percentage
            ),
        )
    )

fig2.update_layout(
    title = dict(
        text = 'Rolling 20-Day Annualised Volatility — GIS Portfolio Holdings',
        font = dict(size=16),
    ),
    xaxis = dict(
        title = 'Date',
        rangeselector = dict(
            buttons = [
                dict(count=1,  label='1M',  step='month', stepmode='backward'),
                dict(count=3,  label='3M',  step='month', stepmode='backward'),
                dict(count=12, label='1Y',  step='month', stepmode='backward'),
                dict(step='all', label='All'),
            ]
        ),
    ),
    yaxis     = dict(title='Annualised Volatility (%)'),
    hovermode = 'x unified',   # unified hover: see all series values at the same date
    height    = 450,
    template  = 'plotly_white',
)

fig2.show()

print()
print('Discussion: Which asset has the highest volatility spikes?')
print('            Are there periods when all assets spike together? What caused them?')

In [ ]:
# ── CHART 3: CORRELATION HEATMAP (PLOTLY) ────────────────────────────────────
# go.Heatmap creates a colour-coded grid where each cell shows the correlation
# between the row asset and the column asset.
# The colour scale runs from red (−1, perfect negative correlation) to green (+1, perfect positive).
# We want LOW correlation between holdings for good diversification.

fig3 = go.Figure(
    data = go.Heatmap(
        z            = corr_matrix.values,                    # the 7×7 matrix of correlation numbers
        x            = corr_matrix.columns.tolist(),          # column labels (ticker names)
        y            = corr_matrix.index.tolist(),            # row labels (ticker names)
        colorscale   = 'RdYlGn',                              # Red = −1 (highly negative), Yellow = 0, Green = +1
        zmin         = -1,                                    # fix colour scale minimum at −1
        zmax         = 1,                                     # fix colour scale maximum at +1
        text         = corr_matrix.values.round(2),           # the numbers shown inside each cell
        texttemplate = '%{text}',                             # display those numbers as text inside each cell
        showscale    = True,                                  # show the colour gradient bar on the right
        colorbar     = dict(title='Correlation'),             # label the colour bar
        hovertemplate = (
            'Row: %{y}<br>'
            'Column: %{x}<br>'
            'Correlation: %{z:.2f}<extra></extra>'            # hover shows the exact correlation value
        ),
    )
)

fig3.update_layout(
    title  = dict(text='Correlation Matrix — GIS Portfolio (Monthly Returns)', font=dict(size=16)),
    height = 500,
    width  = 600,
    template = 'plotly_white',
    xaxis  = dict(title=''),
    yaxis  = dict(title='', autorange='reversed'),   # reverse y-axis: diagonal reads top-left → bottom-right
)

fig3.show()

print()
print('Interactive: hover over any cell to see the exact correlation and which pair it represents.')
print('Click and drag to zoom into a region of the matrix.')
print()
print('What to look for:')
print('  Values near +1.0 (dark green): highly correlated — moving together, weak diversification')
print('  Values near 0 (yellow): uncorrelated — good for diversification')
print('  Values near −1.0 (dark red): negatively correlated — move in opposite directions')
print('  GLD and IEF vs equities: do they provide genuine diversification?')

In [ ]:
# ── CHART 4: 2Y/10Y YIELD SPREAD WITH RECESSION SHADING (PLOTLY) ─────────────
# The 2Y/10Y spread = the 10-year Treasury yield minus the 2-year Treasury yield.
# Normally positive (long rates > short rates).
# When negative (inverted), it has historically preceded recessions by 12–18 months.

fig4 = go.Figure()

# Main spread line
fig4.add_trace(
    go.Scatter(
        x    = spread_series.index,
        y    = spread_series.values,
        name = '2Y/10Y Spread',
        mode = 'lines',
        line = dict(width=2, color='steelblue'),
        hovertemplate = 'Date: %{x|%Y-%m-%d}<br>Spread: %{y:.2f} pp<extra></extra>',
    )
)

# Fill the area below zero in red — visually highlights inversion periods
# clip(upper=0) keeps only negative values; positive values become 0
fig4.add_trace(
    go.Scatter(
        x         = spread_series.index,
        y         = spread_series.clip(upper=0),   # only show values below zero
        fill      = 'tozeroy',                      # fill from this line down to the zero line
        fillcolor = 'rgba(220, 50, 50, 0.3)',       # semi-transparent red
        mode      = 'none',                          # no line — just the fill colour
        name      = 'Inversion (< 0)',
        hoverinfo = 'skip',                          # do not show a hover tooltip for the fill area
    )
)

# Zero reference line — the inversion threshold
fig4.add_hline(
    y                   = 0,
    line_color          = 'black',
    line_width          = 1,
    annotation_text     = 'Zero (inversion threshold)',
    annotation_position = 'bottom right',
)

# Add NBER recession shading using add_vrect (vertical rectangle)
# Each rectangle spans from the recession start date to the end date
for i, (s, e) in enumerate(zip(rec_starts, rec_ends)):
    fig4.add_vrect(
        x0          = str(s.date()),          # left edge: recession start date as a string
        x1          = str(e.date()),          # right edge: recession end date
        fillcolor   = 'grey',
        opacity     = 0.2,                    # 20% opacity — transparent grey shading
        layer       = 'below',                # draw this rectangle behind the chart lines
        line_width  = 0,                      # no border around the rectangle
        annotation_text     = 'Recession' if i == 0 else '',   # label only the first rectangle
        annotation_position = 'top left',
    )

fig4.update_layout(
    title = dict(text='2Y/10Y Yield Spread — Inversions Precede Recessions', font=dict(size=16)),
    xaxis = dict(
        title = 'Date',
        rangeselector = dict(
            buttons = [
                dict(count=1,  label='1Y',  step='year', stepmode='backward'),
                dict(count=3,  label='3Y',  step='year', stepmode='backward'),
                dict(step='all', label='All'),
            ]
        ),
    ),
    yaxis       = dict(title='Spread (percentage points)'),
    height      = 450,
    template    = 'plotly_white',
    showlegend  = True,
)

fig4.show()

In [ ]:
# ── CHART 5: MONTE CARLO FAN CHART (PLOTLY) ──────────────────────────────────
# A fan chart shows the range of possible future outcomes from the Monte Carlo simulation.
# The wider the fan, the more uncertainty. The median is the most likely single path.
# The 5th percentile shows the worst 5% of simulated outcomes — a "stress scenario".

x_days = list(range(1, n_days + 1))   # x-axis: trading days 1 through 252

fig5 = go.Figure()

# Outer band: 5th–95th percentile range (wide, light fill)
# To draw a filled band in Plotly, we go forward along the upper edge, then backward along the lower edge
# This creates a closed polygon that Plotly fills in
fig5.add_trace(
    go.Scatter(
        x         = x_days + x_days[::-1],           # x goes forward (1→252), then backward (252→1)
        y         = list(p95) + list(p5[::-1]),       # upper edge (p95) then lower edge reversed (p5)
        fill      = 'toself',                          # fill the enclosed polygon
        fillcolor = 'rgba(70, 130, 180, 0.15)',        # steelblue at 15% opacity — very light
        mode      = 'none',                             # no line around the band — just the fill
        name      = '5th–95th percentile',
        hoverinfo = 'skip',
    )
)

# Inner band: 25th–75th percentile (narrower, slightly darker fill)
fig5.add_trace(
    go.Scatter(
        x         = x_days + x_days[::-1],
        y         = list(p75) + list(p25[::-1]),
        fill      = 'toself',
        fillcolor = 'rgba(70, 130, 180, 0.30)',        # steelblue at 30% opacity — more visible
        mode      = 'none',
        name      = '25th–75th percentile',
        hoverinfo = 'skip',
    )
)

# Median path — the most likely single outcome
fig5.add_trace(
    go.Scatter(
        x    = x_days,
        y    = list(p50),
        name = 'Median path',
        mode = 'lines',
        line = dict(width=2.5, color='steelblue'),
        hovertemplate = 'Day %{x}<br>Median: %{y:.1f}<extra></extra>',
    )
)

# 5th percentile — the worst-case line (only 5% of paths end below this)
fig5.add_trace(
    go.Scatter(
        x    = x_days,
        y    = list(p5),
        name = '5th percentile (worst 5% of paths)',
        mode = 'lines',
        line = dict(width=1.5, color='crimson', dash='dash'),   # dashed red = downside scenario
        hovertemplate = 'Day %{x}<br>5th pct: %{y:.1f}<extra></extra>',
    )
)

# Starting value reference line at 100
fig5.add_hline(
    y                   = 100,
    line_color          = 'grey',
    line_dash           = 'dot',
    line_width          = 1,
    annotation_text     = 'Start (100)',
    annotation_position = 'bottom right',
)

fig5.update_layout(
    title = dict(
        text = f'Monte Carlo Simulation — GIS Portfolio ({n_sim} Paths, {n_days} Trading Days)',
        font = dict(size=16),
    ),
    xaxis    = dict(title='Trading Days Forward'),
    yaxis    = dict(title='Portfolio Value (Start = 100)'),
    height   = 480,
    template = 'plotly_white',
)

fig5.show()

print()
print(f'After {n_days} trading days (1 year):')
print(f'  5th  percentile: {p5[-1]:.1f}  (worst 5% of simulated outcomes)')
print(f'  Median:          {p50[-1]:.1f}')
print(f'  95th percentile: {p95[-1]:.1f}  (best 5% of simulated outcomes)')

---
## Section 2: The 4-Panel Dashboard

### Why a Dashboard?

Producing 15 separate charts solves a different problem than communicating a portfolio's health. Separate charts require the viewer to hold all of them in memory simultaneously and mentally connect the insights. A portfolio manager opening a risk report should be able to answer four fundamental questions from a single view:

1. **Performance:** How has the portfolio done relative to the benchmark?
2. **Risk:** Has volatility been stable or elevated recently?
3. **Diversification:** Are the holdings genuinely diversified, or moving together?
4. **Macro context:** What is the macro environment telling us about duration and growth?

A 2×2 dashboard answers all four questions without requiring the viewer to navigate between files.

### The 2×2 Grid

| Panel | Top Left | Top Right |
|---|---|---|
| **Content** | Cumulative return (portfolio vs benchmark) | Rolling 20-day volatility |
| **Answers** | Are we ahead of or behind the benchmark? | Is risk elevated or contained right now? |

| Panel | Bottom Left | Bottom Right |
|---|---|---|
| **Content** | Correlation matrix | 2Y/10Y yield spread |
| **Answers** | Are holdings still diversifying each other? | Is the macro backdrop supportive of risk? |

These four together give a comprehensive portfolio health check — the kind of view that would open a weekly portfolio review meeting.

### `make_subplots()` — Plotly's Multi-Panel Builder

`make_subplots()` is Plotly's equivalent of `plt.subplots()` in matplotlib. It creates a grid of panels and allows you to route each trace to a specific panel using `row=` and `col=` arguments.

```python
fig = make_subplots(rows=2, cols=2)     # create a 2×2 grid
fig.add_trace(my_trace, row=1, col=1)   # place this trace in the top-left panel
fig.add_trace(my_trace, row=1, col=2)   # place this trace in the top-right panel
```

The `specs` parameter allows mixing different chart types in the same grid — for example, a heatmap in one panel and a standard line chart in another.

In [ ]:
# ── 4-PANEL DASHBOARD ─────────────────────────────────────────────────────────
# make_subplots() creates a 2-row, 2-column grid of chart panels.
# subplot_titles sets a header for each panel.
# specs defines the type of each panel: 'xy' = standard x/y axis chart, 'heatmap' = heatmap

fig_dashboard = make_subplots(
    rows            = 2,
    cols            = 2,
    subplot_titles  = [
        'Cumulative Return (Base = 100)',
        'Rolling 20-Day Annualised Volatility (%)',
        'Correlation Matrix (Monthly Returns)',
        '2Y/10Y Yield Spread (pp)',
    ],
    specs = [
        [{'type': 'xy'},      {'type': 'xy'}],       # top row: two standard line charts
        [{'type': 'heatmap'}, {'type': 'xy'}],       # bottom row: a heatmap and a line chart
    ],
    vertical_spacing   = 0.12,   # vertical space between the two rows
    horizontal_spacing = 0.08,   # horizontal space between the two columns
)

# ── PANEL 1 (row 1, col 1): Cumulative return ─────────────────────────────────
# Add the GIS portfolio cumulative return line
fig_dashboard.add_trace(
    go.Scatter(
        x          = cumulative_portfolio.index,
        y          = cumulative_portfolio,
        name       = 'GIS Portfolio',
        mode       = 'lines',
        line       = dict(width=2.5, color='steelblue'),
        showlegend = True,   # include this series in the shared dashboard legend
    ),
    row=1, col=1,   # place in the top-left panel
)

# Add the S&P 500 benchmark line to the same panel
fig_dashboard.add_trace(
    go.Scatter(
        x          = cumulative_benchmark.index,
        y          = cumulative_benchmark,
        name       = 'S&P 500',
        mode       = 'lines',
        line       = dict(width=2, color='crimson', dash='dash'),
        showlegend = True,
    ),
    row=1, col=1,
)

# ── PANEL 2 (row 1, col 2): Rolling volatility ───────────────────────────────
# Add one line per holding — suppress individual legends to keep the dashboard clean
for ticker in PORTFOLIO_TICKERS:
    fig_dashboard.add_trace(
        go.Scatter(
            x          = rolling_vol.index,
            y          = rolling_vol[ticker] * 100,   # convert to percentage
            name       = ticker,
            mode       = 'lines',
            line       = dict(width=1.2),
            opacity    = 0.8,
            showlegend = False,    # hide from the legend — too many items would clutter the dashboard
        ),
        row=1, col=2,   # top-right panel
    )

# ── PANEL 3 (row 2, col 1): Correlation heatmap ──────────────────────────────
fig_dashboard.add_trace(
    go.Heatmap(
        z            = corr_matrix.values,
        x            = corr_matrix.columns.tolist(),
        y            = corr_matrix.index.tolist(),
        colorscale   = 'RdYlGn',
        zmin         = -1,
        zmax         = 1,
        text         = corr_matrix.values.round(2),
        texttemplate = '%{text}',
        showscale    = False,    # hide the colour bar inside the dashboard — saves space
        showlegend   = False,
    ),
    row=2, col=1,   # bottom-left panel
)

# ── PANEL 4 (row 2, col 2): 2Y/10Y yield spread ─────────────────────────────
fig_dashboard.add_trace(
    go.Scatter(
        x          = spread_series.index,
        y          = spread_series.values,
        name       = '2Y/10Y Spread',
        mode       = 'lines',
        line       = dict(width=1.8, color='steelblue'),
        showlegend = False,
    ),
    row=2, col=2,   # bottom-right panel
)

# Add zero reference line to the spread panel
fig_dashboard.add_hline(
    y          = 0,
    line_color = 'black',
    line_width = 1,
    row=2, col=2,
)

# Add NBER recession shading to the spread panel
for s, e in zip(rec_starts, rec_ends):
    fig_dashboard.add_vrect(
        x0         = str(s.date()),
        x1         = str(e.date()),
        fillcolor  = 'grey',
        opacity    = 0.2,
        layer      = 'below',
        line_width = 0,
        row=2, col=2,
    )

# ── OVERALL LAYOUT ────────────────────────────────────────────────────────────
fig_dashboard.update_layout(
    title = dict(
        text = 'GIS Portfolio — Investment Health Dashboard',
        font = dict(size=18),
    ),
    height   = 750,
    width    = 1100,
    template = 'plotly_white',
    legend   = dict(orientation='h', y=-0.05),   # horizontal legend below the whole chart grid
)

# Set axis labels for each individual panel
fig_dashboard.update_yaxes(title_text='Value (Base=100)', row=1, col=1)   # top-left y-axis
fig_dashboard.update_yaxes(title_text='Ann. Vol (%)',     row=1, col=2)   # top-right y-axis
fig_dashboard.update_yaxes(title_text='Spread (pp)',      row=2, col=2)   # bottom-right y-axis
fig_dashboard.update_xaxes(title_text='Date',             row=1, col=1)   # top-left x-axis
fig_dashboard.update_xaxes(title_text='Date',             row=1, col=2)   # top-right x-axis
fig_dashboard.update_xaxes(title_text='Date',             row=2, col=2)   # bottom-right x-axis
fig_dashboard.update_yaxes(autorange='reversed',          row=2, col=1)   # heatmap: reverse y-axis

fig_dashboard.show()

print()
print('This is a shareable, interactive dashboard.')
print('Export it as a self-contained HTML file in the next cell.')

In [ ]:
# ── EXPORT TO HTML ────────────────────────────────────────────────────────────
# write_html() saves the interactive dashboard as a single .html file.
# The file is completely self-contained — it includes all the data and interactivity.
# Recipients can open it in any web browser without installing Python or any library.

output_filename = 'gis_dashboard.html'   # the name of the file to create

fig_dashboard.write_html(
    output_filename,
    include_plotlyjs = 'cdn',   # load Plotly JavaScript from the internet (keeps file size small)
                                 # alternative: 'inline' includes Plotly in the file (larger, but works offline)
)

print(f'Dashboard saved as: {output_filename}')
print()
print('To share this file:')
print('  1. In the Colab left sidebar, click the Files icon (folder symbol)')
print(f'  2. Find {output_filename} and right-click → Download')
print('  3. Email the file or share via Teams/SharePoint')
print('  4. Recipients open it in any browser — no Python needed')
print()
print('This is what "a deliverable you could send to a client today" looks like.')

---
### Design Critique Exercise (5 minutes)

Look at the dashboard you just produced. Answer these questions — there are no right answers:

1. **One thing this dashboard communicates well:** ______________________

2. **One thing it fails to communicate** (something a portfolio manager would want to know that is NOT visible): ______________________

3. **I would change ______________________ to improve it** (chart type, colour, title, layout, etc.)

4. **The chart that would add the most value if added as a 5th panel:** ______________________

5. **Is the title "GIS Portfolio — Investment Health Dashboard" conclusion-first? What would a better title be?** ______________________

Share your answers with the group. Facilitator: note the most common additions — these will guide the Claude Code exercise.

---
## Section 3: Claude Code — AI-Assisted Investment Tool Development

### What Claude Code Is

Claude Code is an AI coding assistant built into the terminal. You describe what you want in plain English — "I want a Dash app that shows cumulative returns for a portfolio I type in" — and Claude Code writes the code, runs it, fixes errors, and iterates until the output matches your description.

You direct it the way a senior investment professional directs a junior developer: describe the outcome you want, not the implementation. You say "I need a risk summary table below the CAPM regression" — not "use a `dash_table.DataTable` component with `style_data_conditional` for negative values".

Claude Code is available at [claude.ai/code](https://claude.ai/code) and can be installed as a terminal application. In this seminar, the facilitator will demonstrate it live at the terminal.

---

### The Key Mental Shift

The value you bring is **investment expertise**. You know:
- What Sharpe ratio is appropriate for this mandate and client
- Which risk metrics matter to this investment committee
- What the yield curve is telling you about duration positioning
- Whether a 2% alpha is statistically significant or noise
- How to present drawdown data without alarming a client unnecessarily

**Claude Code does not know any of this.** It only knows how to implement what you describe.

The human is the investment professional. Claude Code is the developer. The better you can describe what a good investment tool looks like, the better the output will be.

---

### Four Rules for an Effective Claude Code Brief

**Rule 1 — Describe the user experience, not the code.**

Not this:
> "Write a Dash app with `dcc.Graph` using `go.Scatter` traces and `dcc.DatePickerRange` for input."

This:
> "Build a tool where I can type comma-separated tickers and see their cumulative returns as an interactive line chart, with a date range picker at the top."

The first brief requires you to already know how Dash works. The second brief requires only that you know what you want the tool to do.

**Rule 2 — Specify the data source.**
> "Use yfinance to download data with `auto_adjust=True`. The tickers are AAPL, MSFT, JPM."

Without this, Claude Code may invent fictional data, use a different library, or ask you to provide a CSV file. Always state: where does the data come from, and what are the specific tickers or FRED codes?

**Rule 3 — Specify the output format.**
- "A Dash app" → an interactive web app running on `localhost:8050`
- "A Plotly chart I can save as HTML" → a single HTML file
- "A pandas DataFrame printed to the console" → no UI, just terminal output

These are very different outputs. Be explicit.

**Rule 4 — Iterate in plain language. One instruction at a time.**
> "Good — now add a summary table below the chart showing annualised return, volatility, and Sharpe ratio for each ticker."

Do not try to specify everything in one giant brief. Start with the core feature, verify it works, then extend. This mirrors how a senior investment professional would actually brief a junior developer.

---

### What Claude Code Does NOT Do

- It cannot tell you whether a Sharpe of 0.4 is acceptable for this client's mandate
- It cannot decide whether to hedge FX exposure in the portfolio
- It cannot interpret what a yield curve inversion means for GIS's specific duration positioning
- It cannot replace the investment judgement that this seminar has been building over three days

Every output Claude Code produces needs to be reviewed by someone with investment expertise before it reaches a client or an investment committee. The tool amplifies your capability; it does not replace your judgement.

---
> ### Facilitator Live Demonstration
>
> **The facilitator will now demonstrate Claude Code live at the terminal.**
>
> Watch the projected screen. The demonstration shows:
> - How to launch Claude Code from the terminal
> - How to give it a brief and watch it write code in real time
> - How to iterate with a follow-up instruction
> - How to read and run the output
>
> **Return to this notebook when the facilitator signals the hands-on activity (Cell 21).**

---

---
## Hands-On Activity — Write Your First Claude Code Brief

**Time: 20 minutes**

Give Claude Code the following instruction (copy it, or adapt it for your own tickers):

---
*"Build a Dash app that does the following:*
*1. Has a text input at the top where I can type comma-separated ticker symbols (e.g. AAPL, MSFT, JPM)*
*2. Has a date range selector (start date and end date)*
*3. Shows an interactive Plotly line chart of cumulative returns for all tickers I type in, using yfinance data*
*4. Below the chart, shows a summary table with four columns: Ticker, Annualised Return, Annualised Volatility, Sharpe Ratio (using the 3-month T-bill as the risk-free rate)*
*5. The chart should have hover labels showing the exact value and date*

*Use yfinance to download the data with auto_adjust=True. The Dash app should run locally and print the URL when it starts."*

---

### Reflection Questions (discuss with your group after the activity):

1. What happened when your instruction was vague? What did Claude Code assume?
2. What follow-up instruction made the biggest improvement to the output?
3. How is this similar to briefing a junior analyst? How is it different?
4. What would you need to specify more precisely to use this in a real client context?

---
## Preview: Afternoon Simulation Challenge (Session 6)

This afternoon you will work in groups to explore and extend one of two ready-made investment tools:

| Challenge | App | Focus |
|-----------|-----|-------|
| **A — Portfolio Viewer** | `portfolio_app.py` | Extend a working Dash app with a new risk or macro panel |
| **B — Momentum Strategy** | `momentum_investing.ipynb` | Interrogate a working backtest; add one new analytical chart |
| **C — Macro Regime Dashboard** | Starter code provided | Build a regime-classification dashboard with Claude Code |

**Group roles:**
- **Investment Lead:** defines what the tool should do and what investment insight it should surface
- **Claude Code Director:** writes and iterates the Claude Code brief
- **QA / Interpreter:** checks output for errors, interprets the results in investment terms
- **Presenter:** structures the 5-minute presentation for the group

**Judging criteria:** Functionality · Analytical Depth · Investment Insight · Presentation · Clarity of Direction

The facilitator will assign challenges and announce group composition after the lunch break.